# Week 2 Practical: Logging Tokens, Latency and Cost for Every Gemini API Call Using MongoDB

- **Platform**: Google Colab / Local Jupyter
- **Programming Language**: Python
- **LLM Provider**: Google Gemini API (`gemini-3.8-flash`)
- **Database**: MongoDB Atlas
- **Level**: Beginner

### 1. Aim of the experiment
Develop a Python application that communicates with Google Gemini API, measures API usage, and stores the details of every API request in MongoDB Atlas.
- Connect to Google Gemini API
- Measure input and output tokens
- Measure latency using `time.perf_counter()`
- Calculate estimated API usage cost ($0.30 / 1M input tokens, $2.50 / 1M output tokens)
- Implement automatic retry with exponential backoff & jitter for temporary failures
- Store usage documents in MongoDB Atlas using upserts
- Query, aggregate, and visualize performance

### Step 2: Install required Python libraries

In [ ]:
!pip install -q -U google-genai pymongo pandas matplotlib

### Step 3 & 4: Configure Gemini API Key
Store your API key in Google Colab Secrets (key: `GEMINI_API_KEY`) or in your local `.env`.

In [ ]:
import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Gemini API key is missing! Please configure GEMINI_API_KEY in Secrets or .env")
print("Gemini API key configured successfully!")

### Step 5 & 6: Create Gemini Client and Test Connection

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(
        timeout=30000
    )
)
MODEL_NAME = "gemini-3.8-flash"
print("Gemini client initialized successfully!")

# Test Gemini API request
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="What is Artificial Intelligence?"
)
print("\n--- Test Gemini Response ---")
print(response.text)

### Step 10 & 11: Configure MongoDB Connection String
Add your MongoDB URI to Colab Secrets (name: `MONGODB_URI`).

In [ ]:
try:
    from google.colab import userdata
    MONGODB_URI = userdata.get("MONGODB_URI")
except Exception:
    MONGODB_URI = os.getenv("MONGODB_URI")

if not MONGODB_URI:
    raise ValueError("MongoDB connection string is missing! Add MONGODB_URI in Secrets or .env")
print("MongoDB credentials loaded successfully!")

### Step 12: Connect Python to MongoDB Atlas

In [ ]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi

mongo_client = MongoClient(
    MONGODB_URI,
    server_api=ServerApi("1"),
    serverSelectionTimeoutMS=5000,
    connectTimeoutMS=5000,
    socketTimeoutMS=10000,
    timeoutMS=10000,
    retryWrites=True
)

# Ping connection
mongo_client.admin.command("ping")

db = mongo_client["GenAI_Monitoring"]
collection = db["api_usage_logs"]
print("MongoDB connected successfully!")
print("Database:", db.name)
print("Collection:", collection.name)

### Step 14 - 16: Pricing, Exponential Backoff Retries, and MongoDB Logging

In [ ]:
import time
import uuid
import random
import pymongo
from datetime import datetime, timezone
from pymongo.errors import PyMongoError

# Gemini 2.5 Flash pricing (USD per 1M tokens)
INPUT_PRICE = 0.30
OUTPUT_PRICE = 2.50

def calculate_cost(input_tokens, output_tokens):
    input_cost = (input_tokens * INPUT_PRICE) / 1_000_000
    output_cost = (output_tokens * OUTPUT_PRICE) / 1_000_000
    return input_cost + output_cost

def retry_delay(attempt):
    delay = min(2 ** (attempt - 1), 16)
    return delay + random.uniform(0, 1)

def is_temporary_error(error):
    status_code = getattr(error, "status_code", getattr(error, "code", None))
    if status_code in [408, 429, 500, 502, 503, 504]:
        return True
    error_name = type(error).__name__.lower()
    temporary_errors = ["timeout", "connection", "temporarily", "unavailable"]
    return any(word in error_name for word in temporary_errors)

def save_log(record):
    for attempt in range(1, 4):
        try:
            with pymongo.timeout(10):
                collection.update_one(
                    {"_id": record["_id"]},
                    {"$setOnInsert": record},
                    upsert=True
                )
            print("Log saved in MongoDB!")
            return True
        except PyMongoError as error:
            print(f"MongoDB error: {error}")
            if attempt == 3:
                print("Unable to save the log.")
                return False
            time.sleep(retry_delay(attempt))
    return False

### Step 17: Main Gemini Monitoring Function (`ask_gemini`)

In [ ]:
def ask_gemini(prompt):
    request_id = str(uuid.uuid4())
    MAX_RETRIES = 4
    for attempt in range(1, MAX_RETRIES + 1):
        start_time = time.perf_counter()
        input_tokens = 0
        output_tokens = 0
        total_tokens = 0
        response_text = ""
        error_message = None
        status = "failed"
        try:
            # Step 1: Send request to Gemini
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )
            # Step 2: Extract response
            response_text = response.text or ""
            # Step 3: Extract tokens
            usage = response.usage_metadata
            if usage:
                input_tokens = usage.prompt_token_count or 0
                output_tokens = usage.candidates_token_count or 0
                total_tokens = usage.total_token_count or 0
            status = "success"
        except Exception as error:
            error_message = str(error)
            print(f"API attempt {attempt} failed: {error_message}")
        
        # Step 5: Measure latency
        latency = time.perf_counter() - start_time
        # Step 6: Calculate cost
        estimated_cost = calculate_cost(input_tokens, output_tokens)
        
        # Step 7: MongoDB document
        log_record = {
            "_id": f"{request_id}_{attempt}",
            "request_id": request_id,
            "timestamp": datetime.now(timezone.utc),
            "model": MODEL_NAME,
            "prompt": prompt,
            "attempt": attempt,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "latency_seconds": round(latency, 3),
            "estimated_cost_usd": round(estimated_cost, 10),
            "status": status,
            "error": error_message
        }
        
        # Step 8: Save to MongoDB
        save_log(log_record)
        
        # Step 9: Print stats
        print("\n========== API MONITORING ==========")
        print("Request ID:", request_id)
        print("Attempt:", attempt)
        print("Status:", status)
        print("Input Tokens:", input_tokens)
        print("Output Tokens:", output_tokens)
        print("Total Tokens:", total_tokens)
        print("Latency:", round(latency, 3), "seconds")
        print("Estimated Cost: $", estimated_cost)
        
        if status == "success":
            print("\n========== GEMINI RESPONSE ==========")
            print(response_text)
            return response_text
            
        if attempt == MAX_RETRIES or not is_temporary_error(error):
            print("\nRequest failed permanently.")
            break
            
        wait_time = retry_delay(attempt)
        print(f"Retrying after {wait_time:.2f} seconds...")
        time.sleep(wait_time)
    return None

### Step 18: Execute Queries dynamically and Log

In [ ]:
sample_questions = [
    "What is Python?",
    "Explain Machine Learning with examples.",
    "What is a Large Language Model?",
    "Explain Generative AI."
]

for q in sample_questions:
    print(f"\n>>> Question: {q}")
    ask_gemini(q)

### Step 19 & 20: Retrieve Records and Tabular Display

In [ ]:
import pandas as pd

records = list(collection.find({}, {"_id": 0}).sort("timestamp", -1))
print("Total API attempts:", len(records))

df = pd.DataFrame(records)
if not df.empty:
    display(df)
else:
    print("No records found.")

### Step 21: Aggregation: Total Tokens, Average Latency, and Estimated Cost

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": None,
            "total_api_attempts": {"$sum": 1},
            "total_input_tokens": {"$sum": "$input_tokens"},
            "total_output_tokens": {"$sum": "$output_tokens"},
            "total_tokens": {"$sum": "$total_tokens"},
            "average_latency": {"$avg": "$latency_seconds"},
            "total_estimated_cost": {"$sum": "$estimated_cost_usd"}
        }
    }
]
result = list(collection.aggregate(pipeline))
if result:
    summary = result[0]
    print("Total API Attempts:", summary["total_api_attempts"])
    print("Total Input Tokens:", summary["total_input_tokens"])
    print("Total Output Tokens:", summary["total_output_tokens"])
    print("Total Tokens:", summary["total_tokens"])
    print("Average Latency:", round(summary["average_latency"], 3), "seconds")
    print("Total Estimated Cost: $", round(summary["total_estimated_cost"], 8))
else:
    print("No API usage data available.")

### Step 22 - 24: Visualizations (Latency, Token Consumption, and Cost Per Attempt)

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(df) + 1), df["latency_seconds"], marker="o", color="b")
    plt.xlabel("API Attempt Number")
    plt.ylabel("Latency (Seconds)")
    plt.title("Gemini API Response Latency")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.bar(range(1, len(df) + 1), df["total_tokens"], color="g")
    plt.xlabel("API Attempt Number")
    plt.ylabel("Total Tokens")
    plt.title("Gemini API Token Consumption")
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.bar(range(1, len(df) + 1), df["estimated_cost_usd"], color="orange")
    plt.xlabel("API Attempt Number")
    plt.ylabel("Estimated Cost (USD)")
    plt.title("Gemini API Cost Per Attempt")
    plt.show()